In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MatplotlibPolygon
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud, Box
from nuscenes.utils.geometry_utils import view_points, transform_matrix
from pyquaternion import Quaternion
import imageio
import shutil

# --- Configuration ---
NUSCENES_VERSION = 'v1.0-mini'  # or 'v1.0-trainval'
NUSCENES_DATAROOT = '/datasets/nuscenes' # ADJUST TO YOUR NUSCENES DATAROOT
LIDAR_SENSOR_NAME = 'LIDAR_TOP'
SCENE_INDEX_TO_PROCESS = 1 # Which scene to make a video of
OUTPUT_VIDEO_FILENAME = f'nuscenes_scene_{SCENE_INDEX_TO_PROCESS}_bev_all_sweeps.mp4'
TEMP_IMAGE_FOLDER = 'temp_bev_frames_all_sweeps'
FRAMES_PER_SECOND = 20 # Video FPS (match LiDAR frequency or adjust for desired speed)

# BEV plot settings
BEV_RANGE_METERS = 50
POINT_SIZE = 0.5
EGO_VEHICLE_COLOR = 'blue'
LIDAR_POINT_COLOR = 'grey'
BOX_EDGE_COLOR = 'red'
BOX_LINE_WIDTH = 1

# --- Helper function to draw a 2D BEV box (same as before) ---
def draw_bev_box(ax, box: Box, color='red', linewidth=1):
    corners_3d = box.corners()
    # Use bottom face corners for BEV polygon
    points_for_polygon = np.transpose(corners_3d[:2, [0, 1, 2, 3]])
    ax.add_patch(MatplotlibPolygon(points_for_polygon, closed=True, fill=False, edgecolor=color, linewidth=linewidth))

    # Draw an arrow for orientation
    center_bottom_face = np.mean(corners_3d[:, [0,1,2,3]], axis=1)
    orientation_vec = box.orientation.rotate(np.array([box.wlh[0] / 2.0, 0, 0]))
    arrow_start = center_bottom_face[:2]
    arrow_end = center_bottom_face[:2] + orientation_vec[:2]
    ax.arrow(arrow_start[0], arrow_start[1],
             arrow_end[0] - arrow_start[0], arrow_end[1] - arrow_start[0],
             head_width=0.5, head_length=0.7, fc=color, ec=color, linewidth=linewidth*0.5)

# --- Main Script ---
if __name__ == '__main__':
    print(f"Loading NuScenes {NUSCENES_VERSION} from {NUSCENES_DATAROOT}...")
    nusc = NuScenes(version=NUSCENES_VERSION, dataroot=NUSCENES_DATAROOT, verbose=False)

    my_scene = nusc.scene[SCENE_INDEX_TO_PROCESS]
    print(f"Processing scene: {my_scene['name']} (Token: {my_scene['token']})")

    if os.path.exists(TEMP_IMAGE_FOLDER):
        shutil.rmtree(TEMP_IMAGE_FOLDER)
    os.makedirs(TEMP_IMAGE_FOLDER)
    print(f"Temporary frames will be saved in: {TEMP_IMAGE_FOLDER}")

    # Start with the first sample's LIDAR_TOP data
    first_sample_token = my_scene['first_sample_token']
    first_sample = nusc.get('sample', first_sample_token)
    current_lidar_data_token = first_sample['data'][LIDAR_SENSOR_NAME]

    frame_count = 0
    fig, ax = plt.subplots(figsize=(10, 10))

    while current_lidar_data_token:
        lidar_data = nusc.get('sample_data', current_lidar_data_token)
        ego_pose_data = nusc.get('ego_pose', lidar_data['ego_pose_token'])
        
        # The sample record (for annotations) is linked from sample_data
        # Annotations are only present for keyframes (is_key_frame=True for lidar_data)
        sample_token_for_annotations = lidar_data['sample_token']
        sample_for_annotations = nusc.get('sample', sample_token_for_annotations)


        # 1. Get LiDAR points in global frame
        pcl_path = os.path.join(nusc.dataroot, lidar_data['filename'])
        pc = LidarPointCloud.from_file(pcl_path)

        calibrated_sensor_data = nusc.get('calibrated_sensor', lidar_data['calibrated_sensor_token'])
        pc.rotate(Quaternion(calibrated_sensor_data['rotation']).rotation_matrix)
        pc.translate(np.array(calibrated_sensor_data['translation']))

        pc.rotate(Quaternion(ego_pose_data['rotation']).rotation_matrix)
        pc.translate(np.array(ego_pose_data['translation']))
        
        points_global = pc.points[:3, :].T

        # 2. Get Ego vehicle position and orientation in global frame
        ego_translation_global = np.array(ego_pose_data['translation'])
        ego_rotation_global = Quaternion(ego_pose_data['rotation'])

        # 3. Plotting
        ax.clear()

        ax.scatter(points_global[:, 0], points_global[:, 1], s=POINT_SIZE, color=LIDAR_POINT_COLOR, alpha=0.7)

        ax.plot(ego_translation_global[0], ego_translation_global[1], 'o', markersize=8, color=EGO_VEHICLE_COLOR, label='Ego Vehicle')
        ego_front_direction = ego_rotation_global.rotate(np.array([2.0, 0, 0]))
        ax.arrow(ego_translation_global[0], ego_translation_global[1],
                 ego_front_direction[0], ego_front_direction[1],
                 head_width=0.8, head_length=1.0, fc=EGO_VEHICLE_COLOR, ec=EGO_VEHICLE_COLOR)

        # # 4. Get and plot annotations (these will only "update" visually at 2Hz keyframes)
        # # We use sample_for_annotations which corresponds to the closest keyframe.
        # for ann_token in sample_for_annotations['anns']:
        #     ann_record = nusc.get('sample_annotation', ann_token)
        #     if 'vehicle' in ann_record['category_name']:
        #         box = Box(center=ann_record['translation'],
        #                   size=ann_record['size'],
        #                   orientation=Quaternion(ann_record['rotation']))
        #         draw_bev_box(ax, box, color=BOX_EDGE_COLOR, linewidth=BOX_LINE_WIDTH)

        ax.set_xlim(ego_translation_global[0] - BEV_RANGE_METERS, ego_translation_global[0] + BEV_RANGE_METERS)
        ax.set_ylim(ego_translation_global[1] - BEV_RANGE_METERS, ego_translation_global[1] + BEV_RANGE_METERS)
        
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel("Global X (meters)")
        ax.set_ylabel("Global Y (meters)")
        ax.set_title(f"Scene: {my_scene['name']} - Sweep: {frame_count} - Timestamp: {lidar_data['timestamp']/1e6:.2f}s "
                     f"(Keyframe: {lidar_data['is_key_frame']})")
        ax.grid(True, linestyle='--', alpha=0.5)
        if frame_count == 0: ax.legend(loc='upper right')

        frame_filename = os.path.join(TEMP_IMAGE_FOLDER, f"frame_{frame_count:04d}.png")
        plt.savefig(frame_filename)
        print(f"Saved {frame_filename} (Timestamp: {lidar_data['timestamp']/1e6:.3f}s, Keyframe: {lidar_data['is_key_frame']})")

        frame_count += 1
        current_lidar_data_token = lidar_data['next'] # Move to the next sweep for this sensor
        # if frame_count > 50: break # For quick testing

    plt.close(fig)
    print(f"\nGenerated {frame_count} frames (all sweeps).")

    print(f"Compiling video: {OUTPUT_VIDEO_FILENAME}...")
    image_files = [os.path.join(TEMP_IMAGE_FOLDER, f) for f in sorted(os.listdir(TEMP_IMAGE_FOLDER)) if f.endswith(".png")]
    
    with imageio.get_writer(OUTPUT_VIDEO_FILENAME, fps=FRAMES_PER_SECOND) as writer:
        for image_file in image_files:
            image = imageio.imread(image_file)
            writer.append_data(image)
    print(f"Video saved as {OUTPUT_VIDEO_FILENAME}")

    # print(f"Cleaning up temporary frames in {TEMP_IMAGE_FOLDER}...")
    # shutil.rmtree(TEMP_IMAGE_FOLDER)
    # print("Done.")